# 01 · Data Pipeline



In [ ]:
import os
import databento as db
import pandas as pd
import numpy as np

# Paths
raw_dir       = "/Users/kylechan/Desktop/Microstructure_Factor_Decay Analysis Across_Liquidity_Regimes/data/raw"
processed_dir = "/Users/kylechan/Desktop/Microstructure_Factor_Decay Analysis Across_Liquidity_Regimes/data/processed"
if not os.path.exists(processed_dir):
    os.makedirs(processed_dir)

def kyle_lambda_vectorized(group):
    """Kyle's Lambda = cov(Δprice, signed_vol) / var(signed_vol) — vectorized OLS slope."""
    dp = group["price"].diff().dropna()
    sv = group["signed_vol"].iloc[1:]
    if len(sv) < 3 or sv.std() == 0:
        return np.nan
    return np.cov(dp, sv)[0, 1] / np.var(sv)

def process_single_file(file_path):
    """Processes one day of MBP-1 data into 5-minute liquidity metrics."""
    store = db.DBNStore.from_file(file_path)
    df    = store.to_df()

    # Filter trades, convert UTC → ET, apply market hours mask
    trades = df[df["action"] == "T"].copy()
    trades.index = trades.index.tz_convert("America/New_York")
    trades = trades.between_time("09:30", "16:00")
    if trades.empty:
        return None

    # Lee-Ready trade classification
    trades["midpoint"] = (trades["bid_px_00"] + trades["ask_px_00"]) / 2
    trades["side_lr"]  = 0
    trades.loc[trades["price"] > trades["midpoint"], "side_lr"] =  1
    trades.loc[trades["price"] < trades["midpoint"], "side_lr"] = -1

    # Tick rule: per-symbol diff so we never compare prices across tickers
    trades["price_diff"] = trades.groupby("symbol")["price"].diff()
    mid_mask = trades["side_lr"] == 0
    trades.loc[mid_mask & (trades["price_diff"] > 0), "side_lr"] =  1
    trades.loc[mid_mask & (trades["price_diff"] < 0), "side_lr"] = -1
    trades["side_lr"]   = trades["side_lr"].replace(0, np.nan).ffill()
    trades["signed_vol"] = trades["side_lr"] * trades["size"]

    # 5-minute aggregation
    trades["effective_spread"] = 2 * np.abs(trades["price"] - trades["midpoint"])
    trades["dollar_vol"]       = trades["price"] * trades["size"]

    resampler = trades.groupby(["symbol", pd.Grouper(freq="5min")])

    metrics = pd.DataFrame()
    metrics["avg_spread"]       = resampler["effective_spread"].mean()
    metrics["total_dollar_vol"] = resampler["dollar_vol"].sum()

    p_start = resampler["price"].first()
    p_end   = resampler["price"].last()
    metrics["return"] = (p_end - p_start) / p_start
    metrics["amihud"] = (np.abs(metrics["return"]) / metrics["total_dollar_vol"]) * 1e6

    metrics["kyle_lambda"] = (
        trades.groupby(["symbol", pd.Grouper(freq="5min")])
        .apply(kyle_lambda_vectorized)
    )

    return metrics.replace([np.inf, -np.inf], np.nan).dropna()

# Process all raw files (skips already-processed days)
all_files = sorted(f for f in os.listdir(raw_dir) if f.endswith(".dbn.zst"))
print(f"Starting pipeline for {len(all_files)} files...")

for filename in all_files:
    raw_path  = os.path.join(raw_dir, filename)
    save_path = os.path.join(processed_dir, filename.replace(".dbn.zst", "_5min.csv"))
    if os.path.exists(save_path):
        print(f"Skipping {filename}, already processed.")
        continue
    print(f"Processing {filename}...")
    try:
        daily = process_single_file(raw_path)
        if daily is not None:
            daily.to_csv(save_path)
    except Exception as e:
        print(f"Error processing {filename}: {e}")

print("ALL FILES PROCESSED!")


In [ ]:
import os
import pandas as pd

processed_dir = "/Users/kylechan/Desktop/Microstructure_Factor_Decay Analysis Across_Liquidity_Regimes/data/processed"
output_path   = "/Users/kylechan/Desktop/Microstructure_Factor_Decay Analysis Across_Liquidity_Regimes/data/master_metrics.csv"

files = sorted(f for f in os.listdir(processed_dir) if f.endswith("_5min.csv"))

if not files:
    print("❌ No processed files found.")
else:
    print(f"🔄 Combining {len(files)} daily files...")
    all_dfs = [pd.read_csv(os.path.join(processed_dir, f)) for f in files]
    master  = pd.concat(all_dfs, ignore_index=True)
    master["ts_recv"] = pd.to_datetime(master["ts_recv"])
    master = master.sort_values(["symbol", "ts_recv"])
    master.to_csv(output_path, index=False)
    print(f"✅ SUCCESS!  {len(master):,} rows → {output_path}")
